In [1]:
import pandas as pd
import zipfile
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Step 1: Specify the path to the zip file containing the CSV
zip_file_path = './20240815_2024-08-16_1608/vtb_20240809.zip'

# Step 2: Open the zip file and read the CSV inside
with zipfile.ZipFile(zip_file_path, 'r') as zip_file:
    # Assuming there's only one CSV file inside the zip
    csv_file_name = zip_file.namelist()[0]
    
    # Read the CSV file, skip the first row, and use the second row as headers
    df = pd.read_csv(zip_file.open(csv_file_name), delimiter=';', skiprows=1, low_memory=False)
    
    # Normalize column names
    df.columns = [c.upper().replace(' ', '_') for c in df.columns]
    
    # Print columns to verify the correct column names
    print("Column names:", df.columns)
    
    # Check if 'DATE' column is present
    if 'DATE' in df.columns:
        df['DATE'] = pd.to_datetime(df['DATE'], format='%d-%m-%Y %H:%M:%S')
        df['EPOCH'] = (df['DATE'] - pd.Timestamp("1970-01-01")) // pd.Timedelta('1s')
        df['DATE_ONLY'] = df['DATE'].dt.date  # Renaming to avoid confusion
        df['TIME_ONLY'] = df['DATE'].dt.time  # Renaming to avoid confusion
    else:
        print("ERROR: 'DATE' column not found")

    # Filter the DataFrame based on instrument series
    instruments = ['F_AKBNK0824', 'F_YKBNK0824', 'F_GARAN0824']
    df = df[df['INSTRUMENT_SERIES'].isin(instruments)]
    df['QUANTITY'] = df.apply(lambda x: -x['QUANTITY'] if x['BUY_SELL'] == 'S' else x['QUANTITY'], axis=1)
    df['datetime_second'] = df['DATE'].dt.floor('S') if 'DATE' in df.columns else None

    # Group by the instrument and datetime_second column and then aggregate
    #df_max = df.groupby(['INSTRUMENT_SERIES', 'datetime_second']).agg({
     #   'PRICE': 'max',
     #   'EXECUTION': lambda x: max(x.abs())  # Applying absolute value before finding the max
    #}).reset_index()


ModuleNotFoundError: No module named 'pandas'

In [ ]:

# Save the modified DataFrame to CSV
output_csv_path = 'C:/viopAnaliz/FUTSPREADS.csv'
df.to_csv(output_csv_path, sep=';', index=False)
print(df.head(25))
